# Solution 2: kNN and neural nets (task 4)
kNN as the reference, then an MLP and a 1D CNN on the raw window: overfit on purpose, then fix it. Validation = unseen people when there are ≥ 3 people, otherwise a random split (which leaks, see task 2).
Needs `uv add pandas` on top of the packages from the task description.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # gestures/load.py (run from gestures/sol/)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from load import load, features, GESTURES, FEATURE_NAMES

# Download the class data once:  curl -o gestures.zip "<export link from the projector>"
SRC = os.environ.get("GESTURES_SRC") or "gestures.zip"
X, y_str, person, meta = load(SRC, return_meta=True)
y = np.array([GESTURES.index(g) for g in y_str])
people = np.unique(person)
n_people = len(people)
print(X.shape, f"{n_people} people")
pd.crosstab(person, y_str).reindex(columns=GESTURES, fill_value=0)

In [ ]:
import time
import torch, torch.nn as nn
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, ShuffleSplit, StratifiedShuffleSplit
torch.set_num_threads(8)

def split(idx, test_size=0.25, seed=0):
    """Hold out whole people if possible."""
    if len(np.unique(person[idx])) >= 3:
        a, b = next(GroupShuffleSplit(1, test_size=test_size, random_state=seed).split(idx, groups=person[idx]))
    elif np.bincount(y[idx])[np.bincount(y[idx]) > 0].min() >= 2:
        a, b = next(StratifiedShuffleSplit(1, test_size=test_size, random_state=seed).split(idx, y[idx]))
    else:   # tiny data: some gesture has a single take
        a, b = next(ShuffleSplit(1, test_size=test_size, random_state=seed).split(idx))
    return idx[a], idx[b]

## kNN reference (before the nets)
Same split as the nets below: validation = unseen people.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from load import features

tr, va = split(np.arange(len(X)))
F = features(X)
ks = [k for k in [1, 3, 5, 15, 50] if k <= len(tr)]
rows = []
for k in ks:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(k)).fit(F[tr], y[tr])
    rows.append((k, m.score(F[tr], y[tr]), m.score(F[va], y[va])))
r = pd.DataFrame(rows, columns=["k", "train acc", "val acc"])
plt.figure(figsize=(5, 3.5))
plt.plot(r.k, r["train acc"], "o-", label="train"); plt.plot(r.k, r["val acc"], "s-", label="val (unseen people)")
plt.xscale("log"); plt.xticks(ks, ks); plt.xlabel("k  (small k = complex model)"); plt.ylabel("accuracy")
plt.ylim(0, 1.02); plt.legend()
r.round(3)

In [ ]:
# scaling and representation, k = 5
k = min(5, len(tr))
mu, sd = X[tr].reshape(-1, 6).mean(0), X[tr].reshape(-1, 6).std(0) + 1e-6
raw = X.reshape(len(X), -1)
raw_std = ((X - mu) / sd).reshape(len(X), -1)
variants = {
    "features, unscaled": (F, KNeighborsClassifier(k)),
    "features, standardized": (F, make_pipeline(StandardScaler(), KNeighborsClassifier(k))),
    "raw window, Euclidean": (raw, KNeighborsClassifier(k)),
    "raw window, per-channel standardized": (raw_std, KNeighborsClassifier(k)),
}
pd.DataFrame([(n, m.fit(Z[tr], y[tr]).score(Z[va], y[va])) for n, (Z, m) in variants.items()],
             columns=["kNN input", "val acc"]).round(3)

## kNN distances
Same split (unseen people), k = 1 so the distance alone decides. DTW on a subsample of the validation takes to keep it to seconds.

In [ ]:
from sklearn.neighbors import NeighborhoodComponentsAnalysis

def dtw_to_all(q, R, band):
    """DTW distance from one series q (T, C) to every series in R (N, T, C), Sakoe-Chiba band."""
    T, N = q.shape[0], len(R)
    D = np.full((N, T + 1, T + 1), np.inf)
    D[:, 0, 0] = 0
    for i in range(1, T + 1):
        for j in range(max(1, i - band), min(T, i + band) + 1):
            cost = ((R[:, j - 1] - q[i - 1]) ** 2).sum(1)
            D[:, i, j] = cost + np.minimum(np.minimum(D[:, i - 1, j], D[:, i, j - 1]), D[:, i - 1, j - 1])
    return np.sqrt(D[:, T, T])

def knn1(Dist, ytr):   # Dist: (n_val, n_train)
    return ytr[Dist.argmin(1)]

Fs = StandardScaler().fit(F[tr]).transform(F)
Z25 = ((X - mu) / sd)[:, ::2]                                   # standardized, 25 Hz: (N, 50, 6)
nrm = np.stack([np.linalg.norm(X[..., :3], axis=-1), np.linalg.norm(X[..., 3:], axis=-1)], -1)
nrm = ((nrm - nrm[tr].mean((0, 1))) / nrm[tr].std((0, 1))).reshape(len(X), -1)
rows = []
for metric in ["euclidean", "manhattan", "cosine", "chebyshev"]:
    acc = KNeighborsClassifier(1, metric=metric).fit(Fs[tr], y[tr]).score(Fs[va], y[va])
    rows.append(("features, standardized", metric, acc))
for metric in ["euclidean", "manhattan", "cosine"]:
    acc = KNeighborsClassifier(1, metric=metric).fit(raw_std[tr], y[tr]).score(raw_std[va], y[va])
    rows.append(("raw window, standardized", metric, acc))
rows.append(("|acc|, |gyro| only (grip-invariant)", "euclidean",
             KNeighborsClassifier(1).fit(nrm[tr], y[tr]).score(nrm[va], y[va])))
nca = make_pipeline(StandardScaler(), NeighborhoodComponentsAnalysis(random_state=0), KNeighborsClassifier(1))
rows.append(("features", "learned (NCA)", nca.fit(F[tr], y[tr]).score(F[va], y[va])))
sub = np.random.default_rng(0).choice(va, min(150, len(va)), replace=False)
t0 = time.time()
for band in [0, 5, 10]:          # band 0 = Euclidean on the same 25 Hz data
    Dist = np.stack([dtw_to_all(Z25[i], Z25[tr], band) for i in sub])
    rows.append((f"raw 25 Hz, {len(sub)} val takes", f"DTW band {band}", (knn1(Dist, y[tr]) == y[sub]).mean()))
print(f"DTW: {time.time() - t0:.0f} s")
pd.DataFrame(rows, columns=["input", "distance", "val acc (1-NN)"]).round(3)

## Models

In [ ]:
class Net(nn.Module):
    def __init__(self, kind, mean, std, dropout=0.0, width=1):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean)); self.register_buffer("std", torch.tensor(std))
        if kind == "mlp":
            h = 256 * width
            self.f = nn.Sequential(nn.Flatten(), nn.Linear(600, h), nn.ReLU(), nn.Dropout(dropout),
                                   nn.Linear(h, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 6))
        else:
            c = 32 * width
            self.f = nn.Sequential(
                nn.Conv1d(6, c, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(c, 2 * c, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
                nn.Conv1d(2 * c, 2 * c, 5, padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1),
                nn.Flatten(), nn.Dropout(dropout), nn.Linear(2 * c, 6))

    def forward(self, x):                      # x: (B, T, 6) in raw units
        x = (x - self.mean) / self.std
        return self.f(x.transpose(1, 2))       # conv wants (B, channels, T)

print(Net("cnn", np.zeros(6, np.float32), np.ones(6, np.float32)))

## Augmentation: noise, time shift, random 3D rotation (a different grip)

In [ ]:
def rand_rot(B, max_deg=30):
    ax = torch.randn(B, 3); ax = ax / ax.norm(dim=1, keepdim=True)
    a = (torch.rand(B) * 2 - 1) * np.deg2rad(max_deg)
    K = torch.zeros(B, 3, 3)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 2] = -ax[:, 2], ax[:, 1], -ax[:, 0]
    K = K - K.transpose(1, 2)
    s, c = torch.sin(a)[:, None, None], torch.cos(a)[:, None, None]
    return torch.eye(3) + s * K + (1 - c) * K @ K           # Rodrigues

def augment(x, rot=True, shift=True, noise=True):
    if rot:
        R = rand_rot(len(x)).transpose(1, 2)
        x = torch.cat([x[..., :3] @ R, x[..., 3:] @ R], -1)   # same rotation for acc and gyro
    if shift:
        x = torch.roll(x, int(np.random.randint(-10, 11)), dims=1)
    if noise:
        x = x + 0.05 * x.std(dim=(0, 1), keepdim=True) * torch.randn_like(x)
    return x

## Training loop with history and optional early stopping

In [ ]:
def train(kind, tr, va, epochs=150, dropout=0.0, wd=0.0, aug=False, early_stop=False, width=1, seed=0):
    torch.manual_seed(seed)
    Xtr = X[tr]
    net = Net(kind, Xtr.reshape(-1, 6).mean(0), Xtr.reshape(-1, 6).std(0) + 1e-6, dropout, width)
    opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=wd)
    lossf = nn.CrossEntropyLoss()
    Xt, yt, Xv, yv = map(torch.tensor, (X[tr], y[tr], X[va], y[va]))
    hist, best = [], (np.inf, None, 0)
    for ep in range(epochs):
        net.train(); perm = torch.randperm(len(Xt)); tl = 0.0
        for i in range(0, len(Xt), 64):
            b = perm[i:i + 64]
            xb = augment(Xt[b]) if aug else Xt[b]
            loss = lossf(net(xb), yt[b])
            opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item() * len(b)
        net.eval()
        with torch.no_grad():
            out = net(Xv); vl = lossf(out, yv).item(); va_acc = (out.argmax(1) == yv).float().mean().item()
        hist.append((tl / len(Xt), vl, va_acc))
        if early_stop and vl < best[0]:
            best = (vl, {k: v.clone() for k, v in net.state_dict().items()}, ep)
    if early_stop:
        net.load_state_dict(best[1])
    return net, np.array(hist), best[2]

def accuracy(net, idx):
    with torch.no_grad():
        return (net.eval()(torch.tensor(X[idx])).argmax(1).numpy() == y[idx]).mean()

def plot_hist(ax, h, title, stop=None):
    ax.plot(h[:, 0], label="train"); ax.plot(h[:, 1], label="val")
    if stop is not None: ax.axvline(stop, color="gray", ls=":", label="early stop")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylim(0, 3); ax.legend()

## Overfit on purpose
Few training people, wide model, many epochs, no regularization.

In [ ]:
tr, va = split(np.arange(len(X)))
few = tr[np.isin(person[tr], np.unique(person[tr])[:2])] if n_people >= 4 else tr[: max(30, len(tr) // 4)]
net, h, _ = train("mlp", few, va, epochs=300, width=4)
fig, ax = plt.subplots(figsize=(6, 4)); plot_hist(ax, h, f"MLP x4, {len(few)} train takes")
print(f"train acc {accuracy(net, few):.2f}   val acc {accuracy(net, va):.2f}")

## MLP vs CNN, plain vs regularized (dropout + weight decay + augmentation + early stopping)

In [ ]:
configs = {
    "mlp plain": dict(kind="mlp"),
    "mlp reg":   dict(kind="mlp", dropout=0.3, wd=1e-2, aug=True, early_stop=True),
    "cnn plain": dict(kind="cnn"),
    "cnn reg":   dict(kind="cnn", dropout=0.3, wd=1e-2, aug=True, early_stop=True),
}
fig, axs = plt.subplots(1, 4, figsize=(20, 4))
rows = []
for ax, (name, cfg) in zip(axs, configs.items()):
    net, h, stop = train(tr=tr, va=va, **cfg)
    plot_hist(ax, h, name, stop if cfg.get("early_stop") else None)
    rows.append((name, accuracy(net, tr), accuracy(net, va)))
plt.tight_layout()
pd.DataFrame(rows, columns=["model", "train acc", "val acc"]).round(3)

## Which fix helps most? (CNN, one change at a time)

In [ ]:
ablation = {
    "none": {}, "dropout": dict(dropout=0.3), "weight decay": dict(wd=1e-2),
    "early stop": dict(early_stop=True), "noise+shift": dict(aug=True),
}
rows = []
for name, cfg in ablation.items():
    net, h, _ = train("cnn", tr, va, epochs=100, **cfg)
    rows.append((name, accuracy(net, va)))
pd.DataFrame(rows, columns=["fix", "val acc"]).round(3)
# note: "noise+shift" here also includes rotation (augment() defaults); edit augment() to separate them

## The number for the class table: CNN (regularized), leave-people-out
Needs ≥ 3 people.

In [ ]:
if n_people >= 3:
    accs = []
    for fold, (trv, te) in enumerate(GroupKFold(min(5, n_people)).split(X, y, person)):
        itr, iva = split(trv, test_size=0.2, seed=fold)
        net, _, _ = train("cnn", itr, iva, dropout=0.3, wd=1e-2, aug=True, early_stop=True, seed=fold)
        accs.append(accuracy(net, te)); print(f"fold {fold}: {accs[-1]:.3f}")
    print(f"CNN by-person accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}, ~{len(itr)} training takes")
else:
    print("need ≥ 3 people")

## Advanced: kNN few-shot baseline for the LLM comparison
1-NN on standardized features, given only m examples per class from the training people (the same examples you would put in the LLM prompt). Mean over 5 random draws.

In [ ]:
rows = []
for m in [1, 3, 5]:
    accs = []
    for seed in range(5):
        r_ = np.random.default_rng(seed)
        shots = np.concatenate([r_.choice(tr[y[tr] == j], min(m, (y[tr] == j).sum()), replace=False)
                                for j in range(6) if (y[tr] == j).any()])
        clf = make_pipeline(StandardScaler(), KNeighborsClassifier(1)).fit(F[shots], y[shots])
        accs.append(clf.score(F[va], y[va]))
    rows.append((m, len(shots), np.mean(accs), np.std(accs)))
pd.DataFrame(rows, columns=["examples per class", "train takes", "val acc", "± std"]).round(3)